# Piping Chains and the RunnablePassthrough Class

In [ ]:
# Run the line of code below to check the version of langchain in the current environment.
# Substitute "langchain" with any other package name to check their version.

In [1]:
pip show langchain

Name: langchain
Version: 0.0.200
Summary: Building applications with LLMs through composability
Home-page: https://www.github.com/hwchase17/langchain
Author: 
Author-email: 
License: MIT
Location: /opt/anaconda3/envs/langchain_env/lib/python3.10/site-packages
Requires: aiohttp, async-timeout, dataclasses-json, langchainplus-sdk, numexpr, numpy, openapi-schema-pydantic, pydantic, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext dotenv
%dotenv

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [4]:
RunnablePassthrough().invoke([1, 2, 3])

[1, 2, 3]

In [6]:
chat_template_tools = ChatPromptTemplate.from_template('''
What are the five most important tools a {job title} needs?
Answer only by listing the tools.
''')

chat_template_strategy = ChatPromptTemplate.from_template('''
Considering the tools provided, develop a strategy for effectively learning and mastering them:
{tools}
''')

In [7]:
chat_template_tools

ChatPromptTemplate(input_variables=['job title'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job title'], template='\nWhat are the five most important tools a {job title} needs?\nAnswer only by listing the tools.\n'))])

In [8]:
chat = ChatOpenAI(model_name = 'gpt-4o-mini', 
                  model_kwargs = {'seed':365},
                  temperature = 0,
                  max_tokens = 100)

In [9]:
string_parser = StrOutputParser()

In [10]:
chain_tools = (chat_template_tools | chat | string_parser | {'tools':RunnablePassthrough()})
chain_strategy = chat_template_strategy | chat | string_parser

In [11]:
print(chain_tools.invoke({'job title':'data scientist'}))

{'tools': '1. Python\n2. R\n3. SQL\n4. Jupyter Notebook\n5. Git'}


In [12]:
print(chain_strategy.invoke({'tools':'''
1. Python
2. R Programming
3. SQL
4. Tableau
5. Hadoop
'''}))

To effectively learn and master Python, R Programming, SQL, Tableau, and Hadoop, you can follow a structured strategy that includes setting clear goals, utilizing various resources, practicing regularly, and applying your knowledge in real-world scenarios. Here’s a step-by-step approach:

### Step 1: Set Clear Goals
- **Define Objectives**: Determine why you want to learn each tool. For example, do you want to analyze data, build machine learning models, or visualize data?
- **Create a


In [13]:
chain_combined = chain_tools | chain_strategy

In [14]:
print(chain_combined.invoke({'job title':'data scientist'}))

To effectively learn and master Python, R, SQL, Jupyter Notebook, and Git, you can follow a structured strategy that incorporates a mix of theoretical understanding, practical application, and project-based learning. Here’s a step-by-step approach:

### 1. Set Clear Goals
- **Define Objectives**: Determine what you want to achieve with each tool. For example, do you want to analyze data, build machine learning models, or manage version control for projects?
- **Timeline**: Set


In [15]:
chain_long = (chat_template_tools | chat | string_parser | {'tools':RunnablePassthrough()} | 
              chat_template_strategy | chat | string_parser)

In [ ]:
print(chain_long)

In [17]:
chain_long.invoke({'job title':'data scientist'})

'To effectively learn and master Python, R, SQL, Jupyter Notebook, and Git, you can follow a structured strategy that incorporates a mix of theoretical understanding, practical application, and project-based learning. Here’s a step-by-step approach:\n\n### 1. Set Clear Goals\n- **Define Objectives**: Determine what you want to achieve with each tool. For example, do you want to analyze data, build machine learning models, or manage version control for projects?\n- **Timeline**: Set'